# HUẤN LUYỆN PRE-TRAINING MÔ HÌNH BASE TRÊN GOOGLE COLAB PRO
## (DeepSeek MLA Architecture + FineWeb-Edu / SmolLM)

Notebook này được thiết kế để chạy trực tiếp trên GPU **A100 (40GB/80GB)** hoặc **L4 (24GB)** của Colab Pro.
Mọi checkpoint sẽ tự động được lưu trữ vào **Google Drive** để chống mất mát khi bị ngắt kết nối.

### 1. Kiểm Tra Phần Cứng GPU & Mount Google Drive

In [21]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
assert torch.cuda.is_available(), "Vui lòng bật GPU trong Runtime -> Change runtime type!"
print(f"GPU hiện tại: {torch.cuda.get_device_name(0)}")
print(f"Dung lượng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Mount Google Drive để lưu checkpoint an toàn
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/R&DTeam/llm_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Thư mục lưu Checkpoint: {CHECKPOINT_DIR}")

PyTorch Version: 2.11.0+cu130
GPU hiện tại: NVIDIA A100-SXM4-80GB
Dung lượng VRAM: 79.25 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục lưu Checkpoint: /content/drive/MyDrive/R&DTeam/llm_checkpoints


### 2. Cài Đặt uv & Kéo Mã Nguồn Về Colab

In [22]:
# 1. Cài đặt công cụ quản lý gói siêu tốc uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

# 2. Kéo mã nguồn trực tiếp từ GitHub repo
!git clone https://github.com/hauiotlink/llm-from-scratch.git /content/llm
%cd /content/llm

# 3. Dùng uv cài đặt dependencies vào hệ thống chỉ trong vài giây
!uv pip install --system -e .

downloading uv 0.13.6 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
fatal: destination path '/content/llm' already exists and is not an empty directory.
/content/llm
Using Python 3.13.16 environment at: /usr
Resolved 72 packages in 87ms                                         
Prepared 1 package in 359ms                                              
Uninstalled 1 package in 0.42ms
Installed 1 package in 0.74ms0 (from file:///content/llm)   
 ~ llm-from-scratch==0.1.0 (from file:///content/llm)


### 3. Cấu Hình Kiến Trúc Mô Hình & Siêu Tham Số Huấn Luyện (Training Config)

In [24]:
from configs.model_config import LLMConfig, AttentionType
from core.model import ModernDecoderOnlyLLM
from core.tokenizer import ByteLevelBPETokenizer

# Nạp Tokenizer đã huấn luyện (hoặc train trên tập ngữ liệu mới)
tokenizer = ByteLevelBPETokenizer.load("data/tokenizer_tinystories.json")
print(f"Vocab Size: {tokenizer.vocab_size}")

# Cấu hình kiến trúc 1 TỶ THAM SỐ (1B Parameters) cho Colab Pro (GPU A100)
config = LLMConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_dim=2048,          # d_model
    num_layers=18,            # 18 transformer layers
    num_heads=16,             # 16 attention heads
    intermediate_dim=6784,    # SwiGLU FFN mở rộng
    max_seq_len=2048,         # Context length 2048
    attention_type=AttentionType.MLA,
    kv_lora_rank=512,         # Nén Latent KV Cache
    qk_rope_head_dim=64,      # Decoupled RoPE Key
    head_dim=128,             # Dimension mỗi head
)

device = torch.device("cuda")
# Kiểm tra bộ nhớ VRAM đang được sử dụng trước khi khởi tạo
allocated_gb = torch.cuda.memory_allocated() / (1024**3)
print(f"VRAM đang dùng trước khi tạo model: {allocated_gb:.2f} GB")
if allocated_gb > 1.0:
    print("⚠️ CẢNH BÁO: VRAM còn dính rác từ session cũ! Đang dọn dẹp...")
    import gc; gc.collect(); torch.cuda.empty_cache()

model = ModernDecoderOnlyLLM(config, device=device)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Khởi tạo mô hình 1B thành công: {total_params:,} tham số ({total_params/1e9:.3f}B)")
print(f"VRAM của mô hình 1B: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")

Vocab Size: 2048


OutOfMemoryError: CUDA out of memory. Tried to allocate 54.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 44.81 MiB is free. Including non-PyTorch memory, this process has 79.20 GiB memory in use. Of the allocated memory 78.58 GiB is allocated by PyTorch, and 122.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

### 4. Nạp Dữ Liệu Stream (FineWeb-Edu hoặc SmolLM) & Khởi Chạy Pre-Training

In [ ]:
from datasets import load_dataset
from data.dataset import create_pretraining_dataloader
from train import train_llm

# Stream tập dữ liệu giáo dục chất lượng cao FineWeb-Edu
print("Đang kết nối luồng dữ liệu HuggingFace...")
ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

# Giải phóng bộ nhớ đệm trước khi nạp dữ liệu
import gc
gc.collect()
torch.cuda.empty_cache()

# Tạo DataLoader với micro-batch size tối ưu cho model 1B (batch=4, accum=16 -> effective batch=64)
dataloader = create_pretraining_dataloader(
    text_stream=ds,
    tokenizer=tokenizer,
    batch_size=4,             # Micro-batch size vừa vặn VRAM
    seq_len=config.max_seq_len,
)

# Khởi chạy Huấn luyện Pre-training 1B Model với Gradient Checkpointing
loss_history = train_llm(
    model=model,
    dataloader=dataloader,
    tokenizer=tokenizer,
    max_steps=5000,           # Số steps huấn luyện mục tiêu
    grad_accum_steps=16,      # Tích lũy 16 steps -> Global batch size = 4 * 16 = 64
    peak_lr=5e-4,
    min_lr=5e-5,
    warmup_steps=100,
    save_interval=100,        # Tự động lưu checkpoint vào Drive mỗi 100 steps
    eval_interval=50,         # Sinh văn bản kiểm tra chất lượng mỗi 50 steps
    checkpoint_dir=CHECKPOINT_DIR,
    resume=True,              # Tự động resume tiếp tục từ checkpoint mới nhất nếu mất mạng
    use_gradient_checkpointing=True, # Tiết kiệm 75% VRAM Activations
    device=device,
)